In [0]:
ric = 'eic'
webos_ver = ['webos22', 'webos23', 'webos24', 'webos25', 'webos26']
plfm_ver = ['webOSTV 22', 'webOSTV 23', 'webOSTV 24', 'webOSTV 25', 'webOSTV 26']
date_ym = '2026-04'

for i in range(0, len(webos_ver)):
    t_webos_ver = webos_ver[i]
    t_plfm_ver  = plfm_ver[i]

    qr = f"""
        select 
            date_ym,
            X_Device_Country as country_code,
            '{t_plfm_ver}' as platform_version,
            DEVICE_NAME as soc,
            normal_log:caller_id,
            normal_log:app_id,
            count(distinct mac_addr) as tv_counters,
            count(mac_addr) as access_counters
        from {ric}_data_ods.tlamp.normal_log_{t_webos_ver}
        where 1=1
        and context_name = 'SAM'
        and message_id = 'NL_APP_LAUNCH_BEGIN'
        and date_ym = '{date_ym}'
        group by
            date_ym,
            X_Device_Country,
            DEVICE_NAME,
            normal_log:caller_id,
            normal_log:app_id
    """

    df = spark.sql(qr)

    df.coalesce(1).write.format("com.databricks.spark.csv")\
        .mode("overwrite").option("header", "true")\
        .save(f's3://s3-lge-he-inbound-{ric}-dev/HEDS/HEDS-1388/{t_webos_ver}')